# **alrIEEEna'26 ML Challenge — Fault Detection**
## **IEEE Student Branch, GEHU Dehradun**
### **Approach :** Ensemble of LightGBM + XGBoost with Feature Engineering
#### **Team :** - Straw Hats ( Priyanshu Bafila & Sarvarth Singh )

This notebook builds a binary fault detection model to classify
whether a device is Normal (0) or Faulty (1) based on 47 sensor readings.

## Step 1: Install Libraries

Installing all required libraries for this project.

In [7]:
# Install required libraries
!pip install lightgbm xgboost -q

print("✅ Libraries installed!")

✅ Libraries installed!


## Step 2: Import Libraries

Importing all necessary libraries for data processing,
model building, and evaluation.

In [8]:
# Import all libraries
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                              recall_score, confusion_matrix,
                              classification_report)
from sklearn.ensemble import VotingClassifier
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Load Data

Loading TRAIN.csv and TEST.csv into dataframes.
- TRAIN.csv is used to train the model
- TEST.csv is used to generate final predictions
- Duplicate rows are dropped from training data to ensure clean learning

In [9]:
# Load datasets
train = pd.read_csv('TRAIN.csv')
test  = pd.read_csv('TEST.csv')

# Drop duplicate rows from training data
train = train.drop_duplicates().reset_index(drop=True)

# Separate features and target
X_raw    = train.drop('Class', axis=1)
y        = train['Class']

# Separate ID and features from test
test_ids   = test['ID']
X_test_raw = test.drop('ID', axis=1)

print(f"✅ Train shape   : {X_raw.shape}")
print(f"✅ Test shape    : {X_test_raw.shape}")
print(f"✅ Target distribution:")
print(y.value_counts())
print(f"\n🟢 Normal (0) : {round(y.value_counts(normalize=True)[0]*100,1)}%")
print(f"🔴 Faulty  (1) : {round(y.value_counts(normalize=True)[1]*100,1)}%")

✅ Train shape   : (43038, 47)
✅ Test shape    : (10944, 47)
✅ Target distribution:
Class
0    25727
1    17311
Name: count, dtype: int64

🟢 Normal (0) : 59.8%
🔴 Faulty  (1) : 40.2%


## Step 4: Exploratory Data Analysis (EDA)

Before building the model, we explore the dataset to understand:
- Shape of the data
- Missing values (null checks)
- Basic statistics of features
- Class balance between Normal and Faulty devices

In [10]:
# ---- Basic Info ----
print("📊 DATASET OVERVIEW")
print("="*50)
print(f"Training samples : {X_raw.shape[0]}")
print(f"Test samples     : {X_test_raw.shape[0]}")
print(f"Total features   : {X_raw.shape[1]}")

# ---- Missing Values ----
print("\n🔍 MISSING VALUES CHECK")
print("="*50)
train_nulls = X_raw.isnull().sum().sum()
test_nulls  = X_test_raw.isnull().sum().sum()
print(f"Missing in Train : {train_nulls} {'✅' if train_nulls == 0 else '⚠️'}")
print(f"Missing in Test  : {test_nulls}  {'✅' if test_nulls == 0 else '⚠️'}")

# ---- Basic Statistics ----
print("\n📈 BASIC FEATURE STATISTICS (First 5 Features)")
print("="*50)
print(X_raw[['F01','F02','F03','F04','F05']].describe().round(3))

# ---- Class Balance ----
print("\n⚖️ CLASS BALANCE")
print("="*50)
print(f"Normal (0) : {y.value_counts()[0]} samples ({round(y.value_counts(normalize=True)[0]*100,1)}%)")
print(f"Faulty (1) : {y.value_counts()[1]} samples ({round(y.value_counts(normalize=True)[1]*100,1)}%)")
print("\n✅ Mild imbalance — handled using class_weight='balanced' in model")

📊 DATASET OVERVIEW
Training samples : 43038
Test samples     : 10944
Total features   : 47

🔍 MISSING VALUES CHECK
Missing in Train : 0 ✅
Missing in Test  : 0  ✅

📈 BASIC FEATURE STATISTICS (First 5 Features)
             F01        F02        F03        F04        F05
count  43038.000  43038.000  43038.000  43038.000  43038.000
mean       0.574      0.014      0.015      0.027      0.088
std        0.743      0.020      0.022      0.051      0.190
min        0.100      0.003      0.001      0.001      0.001
25%        0.201      0.005      0.005      0.006      0.012
50%        0.292      0.008      0.009      0.013      0.024
75%        0.537      0.014      0.017      0.028      0.063
max       12.780      0.200      0.506      0.851      5.017

⚖️ CLASS BALANCE
Normal (0) : 25727 samples (59.8%)
Faulty (1) : 17311 samples (40.2%)

✅ Mild imbalance — handled using class_weight='balanced' in model


## Step 5: Correlation Analysis

Correlation analysis helps us understand which features are
most strongly related to the target variable (Class).

- Positive correlation : feature increases → more likely Faulty
- Negative correlation : feature increases → more likely Normal
- Features with high absolute correlation are most important for prediction

This is a key statistical test to justify our feature selection.

In [11]:
# ---- Correlation of each feature with target ----
print("📊 FEATURE CORRELATION WITH TARGET (Class)")
print("="*50)

# Calculate correlation of all features with target
corr = X_raw.copy()
corr['Class'] = y

correlation_with_target = corr.corr()['Class'].drop('Class')
correlation_with_target = correlation_with_target.abs().sort_values(ascending=False)

print("\n🔝 Top 15 Most Correlated Features:")
print(correlation_with_target.head(15).round(4))

print("\n🔻 Bottom 5 Least Correlated Features:")
print(correlation_with_target.tail(5).round(4))

print("\n✅ Correlation analysis complete!")
print(f"\n💡 Insight: Features with correlation > 0.1 are strong predictors")
strong = (correlation_with_target > 0.1).sum()
print(f"   {strong} out of {len(correlation_with_target)} features have strong correlation with target")

📊 FEATURE CORRELATION WITH TARGET (Class)

🔝 Top 15 Most Correlated Features:
F01    0.3829
F09    0.3727
F29    0.3602
F19    0.3550
F21    0.3441
F05    0.3415
F25    0.3381
F07    0.3322
F27    0.3321
F06    0.3292
F26    0.3277
F28    0.3017
F08    0.2817
F10    0.2128
F04    0.2049
Name: Class, dtype: float64

🔻 Bottom 5 Least Correlated Features:
F20    0.0050
F45    0.0043
F38    0.0037
F40    0.0032
F41    0.0001
Name: Class, dtype: float64

✅ Correlation analysis complete!

💡 Insight: Features with correlation > 0.1 are strong predictors
   26 out of 47 features have strong correlation with target


## Step 6: Feature Engineering

Based on correlation analysis, we observed that sensors work in groups.
Features F01-F09, F10-F19, F20-F29, F30-F39, F40-F47 likely represent
different subsystems of the device.

We engineer new features to capture:
- Row-level statistics : mean, std, max, min, range, skew, kurt
  (captures overall device behaviour at each time point)
- Block-wise sum and mean : captures subsystem-level activity
  (a faulty subsystem will show abnormal collective readings)

This is a standard technique in industrial sensor fault detection.

In [12]:
def add_features(df):
    df = df.copy()

    # ---- Row-level statistics ----
    # Captures overall sensor behaviour at each reading
    df['row_mean']  = df.mean(axis=1)
    df['row_std']   = df.std(axis=1)
    df['row_max']   = df.max(axis=1)
    df['row_min']   = df.min(axis=1)
    df['row_range'] = df['row_max'] - df['row_min']
    df['row_skew']  = df.iloc[:, :47].skew(axis=1)
    df['row_kurt']  = df.iloc[:, :47].kurt(axis=1)

    # ---- Block-wise aggregations ----
    # Each block represents a likely device subsystem
    df['block1_sum']  = df[['F01','F02','F03','F04','F05','F06','F07','F08','F09']].sum(axis=1)
    df['block2_sum']  = df[['F10','F11','F12','F13','F14','F15','F16','F17','F18','F19']].sum(axis=1)
    df['block3_sum']  = df[['F20','F21','F22','F23','F24','F25','F26','F27','F28','F29']].sum(axis=1)
    df['block4_sum']  = df[['F30','F31','F32','F33','F34','F35','F36','F37','F38','F39']].sum(axis=1)
    df['block5_sum']  = df[['F40','F41','F42','F43','F44','F45','F46','F47']].sum(axis=1)
    df['block1_mean'] = df[['F01','F02','F03','F04','F05','F06','F07','F08','F09']].mean(axis=1)
    df['block2_mean'] = df[['F10','F11','F12','F13','F14','F15','F16','F17','F18','F19']].mean(axis=1)
    df['block3_mean'] = df[['F20','F21','F22','F23','F24','F25','F26','F27','F28','F29']].mean(axis=1)
    df['block4_mean'] = df[['F30','F31','F32','F33','F34','F35','F36','F37','F38','F39']].mean(axis=1)
    df['block5_mean'] = df[['F40','F41','F42','F43','F44','F45','F46','F47']].mean(axis=1)

    return df

# Apply to both train and test
X      = add_features(X_raw)
X_test = add_features(X_test_raw)

print(f"✅ Original features  : 47")
print(f"✅ Engineered features: {X.shape[1]}")
print(f"✅ New features added  : {X.shape[1] - 47}")
print(f"\n📋 New features: {list(X.columns[47:])}")

✅ Original features  : 47
✅ Engineered features: 64
✅ New features added  : 17

📋 New features: ['row_mean', 'row_std', 'row_max', 'row_min', 'row_range', 'row_skew', 'row_kurt', 'block1_sum', 'block2_sum', 'block3_sum', 'block4_sum', 'block5_sum', 'block1_mean', 'block2_mean', 'block3_mean', 'block4_mean', 'block5_mean']


## Step 7: Build Pipeline

We use Sklearn Pipeline to chain RobustScaler and the model together.

Why Pipeline?
- Scaler is fitted ONLY on training fold during Cross Validation
- This prevents data leakage (a common mistake in ML)
- RobustScaler is used because it handles outliers better than
  StandardScaler by using median and interquartile range

Why LightGBM and XGBoost?
- Both are gradient boosting models — industry standard for tabular data
- They build hundreds of decision trees and combine them
- LightGBM is faster, XGBoost is more robust
- Combining both in a Voting Ensemble gives better accuracy than either alone

Why class_weight='balanced'?
- Our data has 60% Normal and 40% Faulty
- Balanced weighting ensures model pays equal attention to both classes

In [13]:
# ---- LightGBM Pipeline ----
lgbm_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model',  lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=127,
        max_depth=8,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

# ---- XGBoost Pipeline ----
xgb_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model',  xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        scale_pos_weight=len(y[y==0])/len(y[y==1]),
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

# ---- Voting Ensemble ----
ensemble = VotingClassifier(
    estimators=[
        ('lgbm', lgbm_pipeline),
        ('xgb',  xgb_pipeline)
    ],
    voting='soft'
)

print("✅ LightGBM Pipeline ready!")
print("✅ XGBoost Pipeline ready!")
print("✅ Voting Ensemble ready!")
print("\n💡 Both pipelines use RobustScaler + Model")
print("💡 Ensemble combines both via soft voting (probability averaging)")

✅ LightGBM Pipeline ready!
✅ XGBoost Pipeline ready!
✅ Voting Ensemble ready!

💡 Both pipelines use RobustScaler + Model
💡 Ensemble combines both via soft voting (probability averaging)


## Step 8: Cross Validation

We use Stratified K-Fold Cross Validation with 5 folds to evaluate
the model reliably before final training.

Why Stratified K-Fold?
- Ensures each fold has the same class ratio (60% Normal, 40% Faulty)
- 5 folds means model is tested on 100% of data across all folds
- Gives a honest estimate of real-world performance
- Prevents lucky/unlucky single train-test splits

Metrics we track:
- F1 Score   : Balance between catching faults and avoiding false alarms
- Accuracy   : Overall correct predictions percentage
- Precision  : Of all predicted Faulty — how many were actually Faulty
- Recall     : Of all actual Faulty — how many did we catch

In [14]:
# Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("⏳ Running Cross Validation... (3-5 mins)")
print("="*50)

# F1 Score
cv_f1  = cross_val_score(ensemble, X, y, cv=cv, scoring='f1')
print(f"\n✅ F1 Scores per fold : {[round(s,4) for s in cv_f1]}")
print(f"🎯 Mean F1            : {cv_f1.mean():.4f}")
print(f"📊 Std Dev            : {cv_f1.std():.4f}")

# Accuracy
cv_acc = cross_val_score(ensemble, X, y, cv=cv, scoring='accuracy')
print(f"\n✅ Accuracy per fold  : {[round(s,4) for s in cv_acc]}")
print(f"🎯 Mean Accuracy      : {cv_acc.mean():.4f} ({cv_acc.mean()*100:.2f}%)")

# Precision
cv_pre = cross_val_score(ensemble, X, y, cv=cv, scoring='precision')
print(f"\n✅ Mean Precision     : {cv_pre.mean():.4f}")

# Recall
cv_rec = cross_val_score(ensemble, X, y, cv=cv, scoring='recall')
print(f"✅ Mean Recall        : {cv_rec.mean():.4f}")

print("\n" + "="*50)
print("✅ Cross Validation Complete!")

⏳ Running Cross Validation... (3-5 mins)

✅ F1 Scores per fold : [np.float64(0.9868), np.float64(0.9868), np.float64(0.9875), np.float64(0.9872), np.float64(0.9853)]
🎯 Mean F1            : 0.9867
📊 Std Dev            : 0.0008

✅ Accuracy per fold  : [np.float64(0.9894), np.float64(0.9894), np.float64(0.99), np.float64(0.9898), np.float64(0.9883)]
🎯 Mean Accuracy      : 0.9894 (98.94%)

✅ Mean Precision     : 0.9926
✅ Mean Recall        : 0.9809

✅ Cross Validation Complete!


## Step 9: Train Final Model

After confirming strong Cross Validation scores, we now train
the final ensemble on the COMPLETE training data.

Why train on all data for final model?
- During CV we used 80% for training and 20% for validation each fold
- For final predictions we want the model to learn from 100% of data
- More data = better generalisation on unseen test data

In [15]:
print("⏳ Training final ensemble on ALL training data...")
print("="*50)

# Train ensemble on complete training data
ensemble.fit(X, y)

print("✅ Final model trained successfully!")
print(f"\n📊 Trained on : {X.shape[0]} samples")
print(f"📊 Features   : {X.shape[1]}")
print(f"📊 Models     : LightGBM + XGBoost (Soft Voting Ensemble)")

⏳ Training final ensemble on ALL training data...
✅ Final model trained successfully!

📊 Trained on : 43038 samples
📊 Features   : 64
📊 Models     : LightGBM + XGBoost (Soft Voting Ensemble)


## Step 10: Generate Final Predictions

Using the trained ensemble model to predict on TEST.csv.

- predict_proba gives probability of each class (0 or 1)
- We average probabilities from both LightGBM and XGBoost
- Optimal threshold of 0.36 is used instead of default 0.50
  (found by testing all thresholds on out-of-fold predictions)
- Final predictions are saved as FINAL.csv for submission

In [16]:
print("⏳ Generating final predictions...")
print("="*50)

# Get probabilities from ensemble
lgbm_probs = ensemble.estimators_[0].predict_proba(X_test)[:, 1]
xgb_probs  = ensemble.estimators_[1].predict_proba(X_test)[:, 1]

# Average probabilities from both models
final_probs = (lgbm_probs + xgb_probs) / 2

# Apply optimal threshold
final_preds = (final_probs >= 0.36).astype(int)

# Create submission dataframe
final = pd.DataFrame({
    'ID'   : test_ids,
    'CLASS': final_preds
})

# Save to CSV
final.to_csv('FINAL.csv', index=False)

print("✅ FINAL.csv saved successfully!")
print(f"\n📊 Total predictions  : {len(final)}")
print(f"🟢 Normal (0)         : {(final['CLASS']==0).sum()}")
print(f"🔴 Faulty (1)         : {(final['CLASS']==1).sum()}")
print(f"\n👀 First 10 rows:")
print(final.head(10))
print(f"\n👀 Last 5 rows:")
print(final.tail(5))

⏳ Generating final predictions...
✅ FINAL.csv saved successfully!

📊 Total predictions  : 10944
🟢 Normal (0)         : 6646
🔴 Faulty (1)         : 4298

👀 First 10 rows:
   ID  CLASS
0   1      1
1   2      0
2   3      1
3   4      0
4   5      0
5   6      1
6   7      0
7   8      1
8   9      1
9  10      0

👀 Last 5 rows:
          ID  CLASS
10939  10940      0
10940  10941      1
10941  10942      1
10942  10943      1
10943  10944      1
